# Spatial Mapping, Data Integration, and Numerical Consistency Verification for Engineering Drawings

This notebook is a step-by-step practical course that combines **YOLO11 object detection**, **EasyOCR text extraction**, and **rule-based numerical verification** for engineering drawings.

## Module A - Spatial Mapping & Data Integration
In this module, you will learn how to:
1. Detect engineering symbols (e.g., valves) using a pretrained YOLO11 model.
2. Extract tag numbers and labels using OCR.
3. Associate each text with the most likely symbol using spatial proximity.
4. Integrate extracted information with project and equipment databases.
5. Export structured outputs to CSV and JSON.

## Module B - Numerical Consistency Verification Algorithms
In this module, you will learn how to:
1. Define logic rules for drawing consistency (e.g., same line diameter across sheets).
2. Cross-check extracted values against project references.
3. Detect discrepancies and generate severity-based red flags.
4. Produce QA-ready outputs for engineering review workflows.

> Recommended workflow: Run cells from top to bottom and inspect outputs at each step.

## Step 1 - Environment Setup and Imports

In this section, we install and import all dependencies used in the pipeline:
- `ultralytics`: YOLO11 inference
- `easyocr`: text extraction
- `opencv-python`: image processing
- `pandas`: tabular data handling
- `matplotlib`: visualization

If your environment already has these packages, the install command will just confirm them.

In [ ]:
# If needed, uncomment and run this line once:
# !pip install -q ultralytics easyocr opencv-python pandas matplotlib

import json
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ultralytics import YOLO
import easyocr

print("Libraries imported successfully.")

## Step 2 - Project Paths and Input Validation

We point to:
- pretrained model: `pretrained_yolov11/eng_draw_yolo11m_best.pt` (This is the model that was trained from the previous lesson.)
- sample image: `pretrained_yolov11/154_png_jpg.rf.3857cbb310f9d769ae51bc03c790f677.jpg`

This check is important because incorrect paths are the most common setup issue.

In [ ]:
# Project root = folder where this notebook is located
PROJECT_ROOT = Path.cwd()

MODEL_PATH = PROJECT_ROOT / "pretrained_yolov11" / "eng_draw_yolo11m_best.pt"
IMAGE_PATH = PROJECT_ROOT / "pretrained_yolov11" / "154_png_jpg.rf.3857cbb310f9d769ae51bc03c790f677.jpg"
OUTPUT_DIR = PROJECT_ROOT / "outputs_spatial_mapping"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Model exists:", MODEL_PATH.exists(), "->", MODEL_PATH)
print("Image exists:", IMAGE_PATH.exists(), "->", IMAGE_PATH)
print("Output dir:", OUTPUT_DIR)

## Step 3 - Input Image Loading and Visualization

Before any detection, always visualize the input image.
This helps you:
- confirm that the image is read correctly,
- estimate text size and symbol density,
- choose suitable confidence thresholds later.

In [ ]:
image_bgr = cv2.imread(str(IMAGE_PATH))
if image_bgr is None:
    raise FileNotFoundError(f"Could not read image: {IMAGE_PATH}")

image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(14, 10))
plt.imshow(image_rgb)
plt.title("Input Engineering Drawing")
plt.axis("off")
plt.show()

print("Image shape (H, W, C):", image_rgb.shape)

## YOLO11 Symbol Detection

We use the pretrained model to detect engineering symbols.

Key outputs from YOLO:
- bounding box coordinates (`x1, y1, x2, y2`)
- confidence score
- class ID and class name

These detections become the **symbol candidates** for spatial matching.

In [ ]:
# You can tune this threshold based on your drawing quality
YOLO_CONF = 0.25

# Drawing style controls (easy to customize)
BOX_THICKNESS = 2
TEXT_SCALE = 0.5
TEXT_THICKNESS = 1
SHOW_CONFIDENCE = True

# Per-class color mode: each symbol class gets a different color automatically.
# You can also override specific classes in CLASS_COLOR_OVERRIDES.
USE_PER_CLASS_COLOR = True
CLASS_COLOR_OVERRIDES = {
    # Example:
    # "Gate Valve": (0, 255, 0),
    # "Ball Valve": (255, 0, 0),
}

# Fallback color if per-class mode is off
DEFAULT_BOX_COLOR = (0, 255, 0)   # BGR
DEFAULT_TEXT_COLOR = (255, 255, 255)  # BGR

model = YOLO(str(MODEL_PATH))
yolo_results = model.predict(source=str(IMAGE_PATH), conf=YOLO_CONF, verbose=False)
result = yolo_results[0]

print("YOLO detection completed.")
print("Number of detected symbols:", len(result.boxes))

# Build deterministic color map by class id
class_ids_in_result = sorted({int(b.cls[0].cpu().numpy()) for b in result.boxes}) if len(result.boxes) > 0 else []
class_color_map = {}
for idx, cls_id in enumerate(class_ids_in_result):
    cls_name = result.names[cls_id]

    if cls_name in CLASS_COLOR_OVERRIDES:
        class_color_map[cls_id] = CLASS_COLOR_OVERRIDES[cls_name]
    else:
        # Deterministic pseudo-random color per class id (BGR)
        b = (37 * (idx + 3)) % 256
        g = (97 * (idx + 5)) % 256
        r = (157 * (idx + 7)) % 256
        class_color_map[cls_id] = (b, g, r)

# Draw output with custom style settings
yolo_annotated_bgr = image_bgr.copy()
for box in result.boxes:
    x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int).tolist()
    cls_id = int(box.cls[0].cpu().numpy())
    cls_name = result.names[cls_id]
    conf = float(box.conf[0].cpu().numpy())

    box_color = class_color_map.get(cls_id, DEFAULT_BOX_COLOR) if USE_PER_CLASS_COLOR else DEFAULT_BOX_COLOR
    text_color = box_color if USE_PER_CLASS_COLOR else DEFAULT_TEXT_COLOR

    cv2.rectangle(yolo_annotated_bgr, (x1, y1), (x2, y2), box_color, BOX_THICKNESS)

    label = f"{cls_name} {conf:.2f}" if SHOW_CONFIDENCE else cls_name
    text_y = max(15, y1 - 6)
    cv2.putText(
        yolo_annotated_bgr,
        label,
        (x1, text_y),
        cv2.FONT_HERSHEY_SIMPLEX,
        TEXT_SCALE,
        text_color,
        TEXT_THICKNESS,
        cv2.LINE_AA,
    )

yolo_annotated_rgb = cv2.cvtColor(yolo_annotated_bgr, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(14, 10))
plt.imshow(yolo_annotated_rgb)
plt.title("YOLO11 Detection Output (Per-Class Colors)")
plt.axis("off")
plt.show()

# Save detection output image
yolo_output_image_path = OUTPUT_DIR / "yolo11_detection_output.jpg"
cv2.imwrite(str(yolo_output_image_path), yolo_annotated_bgr)
print("Saved YOLO output image:", yolo_output_image_path)

# Optional: preview class-color mapping
if USE_PER_CLASS_COLOR and len(class_color_map) > 0:
    color_legend_df = pd.DataFrame(
        [
            {
                "class_id": cid,
                "class_name": result.names[cid],
                "bgr_color": class_color_map[cid],
            }
            for cid in class_ids_in_result
        ]
    )
    display(color_legend_df)

In [ ]:
symbol_records = []

for i, box in enumerate(result.boxes):
    x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().tolist()
    conf = float(box.conf[0].cpu().numpy())
    class_id = int(box.cls[0].cpu().numpy())
    class_name = result.names[class_id]

    cx = (x1 + x2) / 2
    cy = (y1 + y2) / 2

    symbol_records.append(
        {
            "symbol_id": i,
            "class_id": class_id,
            "class_name": class_name,
            "confidence": round(conf, 4),
            "x1": float(x1),
            "y1": float(y1),
            "x2": float(x2),
            "y2": float(y2),
            "center_x": float(cx),
            "center_y": float(cy),
            "width": float(x2 - x1),
            "height": float(y2 - y1),
        }
    )

symbols_df = pd.DataFrame(symbol_records)
symbols_df.head(10)

## Step 5 - EasyOCR Text Extraction

EasyOCR returns text boxes with confidence scores.

For each OCR text box, we will compute:
- cleaned text
- text center point (`center_x`, `center_y`)

These text centers are used for distance-based symbol association.

In [ ]:
reader = easyocr.Reader(["en"], gpu=False)
ocr_results = reader.readtext(image_rgb)

print("OCR text boxes found:", len(ocr_results))

# Draw OCR output: polygon boxes + recognized text
ocr_viz = image_rgb.copy()
for bbox, text, conf in ocr_results:
    pts = np.array(bbox, dtype=np.int32)

    # Draw polygon around detected text
    cv2.polylines(ocr_viz, [pts], isClosed=True, color=(255, 0, 0), thickness=2)

    # Put recognized text near the first polygon point
    anchor_x, anchor_y = int(pts[0][0]), int(pts[0][1])
    label = f"{str(text)} ({float(conf):.2f})"
    cv2.putText(
        ocr_viz,
        label,
        (anchor_x, max(15, anchor_y - 5)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.45,
        (255, 255, 0),
        1,
        cv2.LINE_AA,
    )

plt.figure(figsize=(14, 10))
plt.imshow(ocr_viz)
plt.title("EasyOCR Text Detection Output")
plt.axis("off")
plt.show()

# Save OCR output image
ocr_output_image_path = OUTPUT_DIR / "easyocr_text_output.jpg"
cv2.imwrite(str(ocr_output_image_path), cv2.cvtColor(ocr_viz, cv2.COLOR_RGB2BGR))
print("Saved OCR output image:", ocr_output_image_path)

### Combined Visualization - OCR Text + YOLO Symbol Detection

This view overlays both:
- YOLO symbol bounding boxes
- EasyOCR text polygons and extracted text labels

Use this combined output for quick visual QA before spatial matching.

In [ ]:
# Combined overlay: symbols + OCR text on one image
combined_viz_bgr = image_bgr.copy()

# 1) Draw YOLO symbol detection
for box in result.boxes:
    x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int).tolist()
    cls_id = int(box.cls[0].cpu().numpy())
    cls_name = result.names[cls_id]
    conf = float(box.conf[0].cpu().numpy())

    # Keep color behavior consistent with Step 4 if available
    if "USE_PER_CLASS_COLOR" in globals() and USE_PER_CLASS_COLOR and "class_color_map" in globals():
        box_color = class_color_map.get(cls_id, (0, 255, 0))
    else:
        box_color = (0, 255, 0)

    cv2.rectangle(combined_viz_bgr, (x1, y1), (x2, y2), box_color, 2)

    yolo_label = f"{cls_name} {conf:.2f}"
    cv2.putText(
        combined_viz_bgr,
        yolo_label,
        (x1, max(15, y1 - 6)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.45,
        box_color,
        1,
        cv2.LINE_AA,
    )

# 2) Draw OCR text extraction
for bbox, text, conf in ocr_results:
    pts = np.array(bbox, dtype=np.int32)
    cv2.polylines(combined_viz_bgr, [pts], isClosed=True, color=(255, 0, 0), thickness=2)

    anchor_x, anchor_y = int(pts[0][0]), int(pts[0][1])
    ocr_label = f"{str(text)} ({float(conf):.2f})"
    cv2.putText(
        combined_viz_bgr,
        ocr_label,
        (anchor_x, max(15, anchor_y - 5)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.45,
        (0, 255, 255),
        1,
        cv2.LINE_AA,
    )

combined_viz_rgb = cv2.cvtColor(combined_viz_bgr, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(16, 12))
plt.imshow(combined_viz_rgb)
plt.title("Combined Output: YOLO Symbols + EasyOCR Text")
plt.axis("off")
plt.show()

combined_output_image_path = OUTPUT_DIR / "combined_yolo_ocr_output.jpg"
cv2.imwrite(str(combined_output_image_path), combined_viz_bgr)
print("Saved combined output image:", combined_output_image_path)

In [ ]:
def polygon_center(points):
    xs = [p[0] for p in points]
    ys = [p[1] for p in points]
    return float(np.mean(xs)), float(np.mean(ys))

text_records = []
for i, (bbox, text, conf) in enumerate(ocr_results):
    clean_text = str(text).strip()
    if not clean_text:
        continue

    cx, cy = polygon_center(bbox)

    x_coords = [p[0] for p in bbox]
    y_coords = [p[1] for p in bbox]

    text_records.append(
        {
            "text_id": i,
            "text": clean_text,
            "ocr_confidence": float(conf),
            "text_x1": float(min(x_coords)),
            "text_y1": float(min(y_coords)),
            "text_x2": float(max(x_coords)),
            "text_y2": float(max(y_coords)),
            "center_x": cx,
            "center_y": cy,
        }
    )

texts_df = pd.DataFrame(text_records)
texts_df.head(15)

## Step 6 - Spatial Matching Logic (Nearest Symbol Association)

### Matching logic
For each text center point:
1. Compute Euclidean distance to every symbol center.
2. Pick the nearest symbol.
3. Accept the match only if distance is less than a configurable threshold (`MAX_DISTANCE`).

This gives a practical baseline for **text-symbol association** in engineering drawings.

### Step 6 Output Visualization - Text-Symbol Associations

This visualization shows the association result directly on the drawing:
- green boxes: detected symbols,
- yellow dots/text: OCR text centers,
- cyan lines: matched text -> symbol links.

Use this image to quickly validate whether nearest-neighbor matching is correct.

In [ ]:
MAX_DISTANCE = 180  # pixels, tune based on drawing scale and text spacing

def euclidean_distance(x1, y1, x2, y2):
    return float(np.sqrt((x1 - x2) ** 2 + (y1 - y2) ** 2))

association_records = []

if symbols_df.empty:
    print("No symbols detected by YOLO. Try reducing YOLO_CONF.")
elif texts_df.empty:
    print("No text detected by OCR. Try image preprocessing or OCR settings.")
else:
    for _, t in texts_df.iterrows():
        tx, ty = t["center_x"], t["center_y"]

        temp = symbols_df.copy()
        temp["distance"] = temp.apply(
            lambda s: euclidean_distance(tx, ty, s["center_x"], s["center_y"]), axis=1
        )

        nearest = temp.sort_values("distance", ascending=True).iloc[0]
        distance_val = float(nearest["distance"])

        is_matched = distance_val <= MAX_DISTANCE

        association_records.append(
            {
                "text_id": int(t["text_id"]),
                "text": t["text"],
                "ocr_confidence": float(t["ocr_confidence"]),
                "text_center_x": float(tx),
                "text_center_y": float(ty),
                "matched": bool(is_matched),
                "distance_to_symbol": round(distance_val, 2),
                "symbol_id": int(nearest["symbol_id"]) if is_matched else None,
                "symbol_class": nearest["class_name"] if is_matched else None,
                "symbol_confidence": float(nearest["confidence"]) if is_matched else None,
                "symbol_center_x": float(nearest["center_x"]) if is_matched else None,
                "symbol_center_y": float(nearest["center_y"]) if is_matched else None,
            }
        )

associations_df = pd.DataFrame(association_records)
associations_df.head(20)

In [ ]:
matched_count = int(associations_df["matched"].sum()) if not associations_df.empty else 0
total_texts = len(associations_df)

print(f"Total OCR texts: {total_texts}")
print(f"Matched texts: {matched_count}")
print(f"Unmatched texts: {total_texts - matched_count}")

if total_texts > 0:
    print(f"Match ratio: {matched_count / total_texts:.2%}")

## Step 7 - Association Visualization on Drawing

Visualization helps validate whether proximity matching is reasonable.

Legend:
- Green box: detected symbol
- Yellow text: OCR result
- Cyan line: matched pair (text -> symbol center)

If many lines look wrong, adjust:
- `YOLO_CONF`
- `MAX_DISTANCE`
- image preprocessing quality

In [ ]:
viz = image_rgb.copy()

# Draw symbols
for _, s in symbols_df.iterrows():
    x1, y1, x2, y2 = int(s["x1"]), int(s["y1"]), int(s["x2"]), int(s["y2"])
    cv2.rectangle(viz, (x1, y1), (x2, y2), (0, 255, 0), 2)

# Draw text centers and match lines
if not associations_df.empty:
    for _, a in associations_df.iterrows():
        tx, ty = int(a["text_center_x"]), int(a["text_center_y"])
        cv2.circle(viz, (tx, ty), 3, (255, 255, 0), -1)
        cv2.putText(
            viz,
            str(a["text"]),
            (tx + 4, ty - 4),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.45,
            (255, 255, 0),
            1,
            cv2.LINE_AA,
        )

        if bool(a["matched"]):
            sx, sy = int(a["symbol_center_x"]), int(a["symbol_center_y"])
            cv2.line(viz, (tx, ty), (sx, sy), (0, 255, 255), 1)

plt.figure(figsize=(16, 12))
plt.imshow(viz)
plt.title("YOLO + EasyOCR Spatial Association")
plt.axis("off")
plt.show()

## Step 8 - Integrated Result Table Construction

This table is your final structured dataset for downstream workflows (QA/QC, digital twin pipelines, BOM mapping, etc.).

Each row represents an OCR text and (if matched) its linked engineering symbol.

In [ ]:
final_df = associations_df.copy()

# Optional: keep only useful columns and order them
final_columns = [
    "text_id",
    "text",
    "ocr_confidence",
    "matched",
    "distance_to_symbol",
    "symbol_id",
    "symbol_class",
    "symbol_confidence",
    "text_center_x",
    "text_center_y",
    "symbol_center_x",
    "symbol_center_y",
]

final_df = final_df[final_columns]
final_df.head(20)

## Step 9 - Structured Export (CSV and JSON)

We export in two formats:
- **CSV**: easy for Excel and quick analysis
- **JSON**: suitable for APIs and system integration

In [ ]:
csv_path = OUTPUT_DIR / "spatial_mapping_results.csv"
json_path = OUTPUT_DIR / "spatial_mapping_results.json"

final_df.to_csv(csv_path, index=False, encoding="utf-8")
final_df.to_json(json_path, orient="records", force_ascii=False, indent=2)

print("Export complete:")
print("CSV :", csv_path)
print("JSON:", json_path)

## Step 10 - Optional Quality Improvements

To improve mapping quality in real projects, you can add:
1. **Image preprocessing before OCR** (binarization, denoising, contrast enhancement).
2. **Class filtering** (e.g., only valve classes) before matching.
3. **Directional heuristics** (text tends to appear left/right of symbol in your template style).
4. **One-to-many / many-to-one logic** for compound labels.
5. **Graph-based matching** if simple nearest-neighbor is not enough.

These are excellent follow-up exercises for this course.

### Optional Valve-Focused Visualization

This visualization highlights only valve-focused matching:
- green boxes: detected valve symbols,
- yellow text: OCR text,
- cyan line: text linked to nearest valve.

Use this image to quickly verify whether valve-only association is reasonable.

In [ ]:
# Visualize optional valve-focused analysis and save image
valve_viz = image_rgb.copy()

# Draw only valve boxes
for _, s in valves_df.iterrows():
    x1, y1, x2, y2 = int(s["x1"]), int(s["y1"]), int(s["x2"]), int(s["y2"])
    cv2.rectangle(valve_viz, (x1, y1), (x2, y2), (0, 255, 0), 2)

# Draw text points and nearest-valve links
if "valve_assoc_df" in globals() and not valve_assoc_df.empty:
    for _, t in texts_df.iterrows():
        tx, ty = int(t["center_x"]), int(t["center_y"])
        cv2.circle(valve_viz, (tx, ty), 3, (255, 255, 0), -1)
        cv2.putText(
            valve_viz,
            str(t["text"]),
            (tx + 4, ty - 4),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.45,
            (255, 255, 0),
            1,
            cv2.LINE_AA,
        )

        # nearest valve for this text
        if not valves_df.empty:
            temp = valves_df.copy()
            temp["distance"] = temp.apply(
                lambda s: euclidean_distance(t["center_x"], t["center_y"], s["center_x"], s["center_y"]), axis=1
            )
            nearest = temp.sort_values("distance", ascending=True).iloc[0]
            if float(nearest["distance"]) <= MAX_DISTANCE:
                sx, sy = int(nearest["center_x"]), int(nearest["center_y"])
                cv2.line(valve_viz, (tx, ty), (sx, sy), (0, 255, 255), 1)

plt.figure(figsize=(16, 12))
plt.imshow(valve_viz)
plt.title("Optional Valve-Focused Analysis Output")
plt.axis("off")
plt.show()

valve_output_image_path = OUTPUT_DIR / "optional_valve_focused_output.jpg"
cv2.imwrite(str(valve_output_image_path), cv2.cvtColor(valve_viz, cv2.COLOR_RGB2BGR))
print("Saved optional valve-focused output image:", valve_output_image_path)

## End of Lab

You have completed a full mini pipeline for **Spatial Mapping & Data Integration** on engineering drawings:
- detection (YOLO11),
- OCR (EasyOCR),
- spatial association,
- structured export.

Next step suggestion:
- run this notebook on multiple drawings,
- compare match ratios,
- tune `YOLO_CONF` and `MAX_DISTANCE` systematically.

## Optional Valve-Focused Analysis

If your lesson objective is specifically *"Which text belongs to which valve?"*, filter detections to classes that contain `Valve` in their class name and rerun matching with this subset.

In [ ]:
valves_df = symbols_df[symbols_df["class_name"].str.contains("Valve", case=False, na=False)].copy()
print("Detected valve symbols:", len(valves_df))

if valves_df.empty:
    print("No valve classes detected in this image.")
else:
    valve_assoc_records = []
    for _, t in texts_df.iterrows():
        tx, ty = t["center_x"], t["center_y"]

        temp = valves_df.copy()
        temp["distance"] = temp.apply(
            lambda s: euclidean_distance(tx, ty, s["center_x"], s["center_y"]), axis=1
        )
        nearest = temp.sort_values("distance", ascending=True).iloc[0]

        valve_assoc_records.append(
            {
                "text": t["text"],
                "nearest_valve_class": nearest["class_name"],
                "distance": float(nearest["distance"]),
                "matched_by_threshold": float(nearest["distance"]) <= MAX_DISTANCE,
            }
        )

    valve_assoc_df = pd.DataFrame(valve_assoc_records)
    valve_assoc_df.head(20)

### Optional Valve-Focused Visualization

This visualization is dedicated to valve-focused analysis:
- green boxes: detected valve symbols,
- yellow text points: OCR text,
- cyan lines: text linked to nearest valve (within threshold).

It helps confirm whether valve-only association is reasonable before rule checks.

In [ ]:
# Visualize Optional Valve-Focused Analysis and save image
valve_viz = image_rgb.copy()

# Draw detected valve boxes
if "valves_df" in globals() and not valves_df.empty:
    for _, s in valves_df.iterrows():
        x1, y1, x2, y2 = int(s["x1"]), int(s["y1"]), int(s["x2"]), int(s["y2"])
        cv2.rectangle(valve_viz, (x1, y1), (x2, y2), (0, 255, 0), 2)

# Draw OCR text points and nearest valve links
if "texts_df" in globals() and not texts_df.empty and "valves_df" in globals() and not valves_df.empty:
    for _, t in texts_df.iterrows():
        tx, ty = int(t["center_x"]), int(t["center_y"])
        cv2.circle(valve_viz, (tx, ty), 3, (255, 255, 0), -1)
        cv2.putText(
            valve_viz,
            str(t["text"]),
            (tx + 4, ty - 4),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.45,
            (255, 255, 0),
            1,
            cv2.LINE_AA,
        )

        # nearest valve for each text
        temp = valves_df.copy()
        temp["distance"] = temp.apply(
            lambda s: euclidean_distance(t["center_x"], t["center_y"], s["center_x"], s["center_y"]), axis=1
        )
        nearest = temp.sort_values("distance", ascending=True).iloc[0]

        if float(nearest["distance"]) <= MAX_DISTANCE:
            sx, sy = int(nearest["center_x"]), int(nearest["center_y"])
            cv2.line(valve_viz, (tx, ty), (sx, sy), (0, 255, 255), 1)

plt.figure(figsize=(16, 12))
plt.imshow(valve_viz)
plt.title("Optional Valve-Focused Analysis Output")
plt.axis("off")
plt.show()

valve_output_image_path = OUTPUT_DIR / "optional_valve_focused_output.jpg"
cv2.imwrite(str(valve_output_image_path), cv2.cvtColor(valve_viz, cv2.COLOR_RGB2BGR))
print("Saved optional valve-focused output image:", valve_output_image_path)

### Scenario Foundation - Standardized Tag Extraction

Most project databases use normalized tag formats. We first parse tag-like patterns from OCR text.

Example tags:
- `V-101`
- `P-210A`
- `LV-12`

You can customize the regular expression for your project naming convention.

In [ ]:
import re

TAG_PATTERN = re.compile(r"\b[A-Z]{1,4}-?\d{1,4}[A-Z]?\b")

def extract_tag(text: str):
    text = str(text).upper().strip()
    m = TAG_PATTERN.search(text)
    return m.group(0) if m else None

final_df["extracted_tag"] = final_df["text"].apply(extract_tag)
final_df[["text", "extracted_tag", "symbol_class", "matched"]].head(25)

In [ ]:
PROJECT_DB_PATH = PROJECT_ROOT / "data" / "project_equipment_database.csv"

if PROJECT_DB_PATH.exists():
    project_db = pd.read_csv(PROJECT_DB_PATH)
    print("Loaded real project database:", PROJECT_DB_PATH)
else:
    # Demo data for training; replace with your real project DB file
    project_db = pd.DataFrame(
        [
            {"tag": "V-101", "expected_type": "Gate Valve", "service": "Steam", "line_id": "L-1001", "sheet": "SHT-01", "expected_diameter_mm": 100},
            {"tag": "V-102", "expected_type": "Ball Valve", "service": "Water", "line_id": "L-1002", "sheet": "SHT-01", "expected_diameter_mm": 80},
            {"tag": "P-210A", "expected_type": "Centrifugal Pump", "service": "Feed", "line_id": "L-2001", "sheet": "SHT-02", "expected_diameter_mm": 150},
            {"tag": "LV-12", "expected_type": "Globe Valve", "service": "Control", "line_id": "L-3004", "sheet": "SHT-03", "expected_diameter_mm": 50},
        ]
    )
    print("Using demo project database (no real CSV found).")

# Normalize join keys
project_db["tag_norm"] = project_db["tag"].astype(str).str.upper().str.strip()
final_df["tag_norm"] = final_df["extracted_tag"].astype(str).str.upper().str.strip()

# Keep only rows with extracted tags for DB matching
observed_df = final_df[final_df["extracted_tag"].notna()].copy()

type_matched_df = observed_df.merge(
    project_db,
    how="left",
    left_on="tag_norm",
    right_on="tag_norm",
    suffixes=("", "_db"),
)

type_matched_df["type_match_ok"] = (
    type_matched_df["expected_type"].astype(str).str.lower().str.strip()
    == type_matched_df["symbol_class"].astype(str).str.lower().str.strip()
)

type_matched_df[[
    "text", "extracted_tag", "symbol_class", "expected_type", "type_match_ok", "service", "line_id", "sheet"
]].head(25)

### Scenario Foundation - Logic Rule Engine

Rule example from project standards:
- **R1: Line diameter consistency across sheets**
  - If the same `line_id` appears in multiple sheets, `expected_diameter_mm` must remain consistent.

You can encode many engineering checks in this rule engine.

In [ ]:
rule_outputs = [
    rule_line_diameter_consistency(type_matched_df),
    rule_missing_project_tag(type_matched_df),
    rule_symbol_type_mismatch(type_matched_df),
]

issues_df = pd.concat(rule_outputs, ignore_index=True) if rule_outputs else pd.DataFrame()

if issues_df.empty:
    print("No rule violations found.")
else:
    print("Rule violations found:", len(issues_df))
    display(issues_df.head(30))

In [ ]:
EQUIP_LIST_PATH = PROJECT_ROOT / "data" / "external_equipment_list.csv"

if EQUIP_LIST_PATH.exists():
    equip_df = pd.read_csv(EQUIP_LIST_PATH)
    print("Loaded external equipment list:", EQUIP_LIST_PATH)
else:
    equip_df = pd.DataFrame(
        [
            {"tag": "V-101", "equipment_status": "ACTIVE", "vendor": "VendorA", "criticality": "HIGH"},
            {"tag": "V-102", "equipment_status": "DECOMMISSIONED", "vendor": "VendorB", "criticality": "MEDIUM"},
            {"tag": "P-210A", "equipment_status": "ACTIVE", "vendor": "VendorC", "criticality": "HIGH"},
        ]
    )
    print("Using demo external equipment list (no real CSV found).")

equip_df["tag_norm"] = equip_df["tag"].astype(str).str.upper().str.strip()

xref_df = type_matched_df.merge(
    equip_df[["tag_norm", "equipment_status", "vendor", "criticality"]],
    how="left",
    on="tag_norm",
)

xref_df[[
    "extracted_tag", "symbol_class", "expected_type", "equipment_status", "vendor", "criticality"
]].head(25)

### Scenario Foundation - Automated Discrepancy and Red-Flag Detection

We combine all checks into a single red-flag dashboard dataset.

Sample red-flag logic:
- `CRITICAL`: Tag exists but equipment is `DECOMMISSIONED`.
- `HIGH`: Type mismatch (detected vs expected).
- `MEDIUM`: Missing project DB record.
- `LOW`: Missing external equipment cross-reference.

In [ ]:
# Quick summary for reporting
if red_flags_df.empty:
    print("No red-flag summary to report.")
else:
    summary_df = (
        red_flags_df.groupby(["severity", "flag_code"], observed=False)
        .size()
        .reset_index(name="count")
        .sort_values(["severity", "flag_code"])
    )
    display(summary_df)

### Scenario Foundation - Export QA Outputs for Project Workflows

We export:
- integrated cross-reference table,
- logic rule violations,
- red-flag list.

These files can be consumed by dashboards, BI tools, or engineering review meetings.

## Practical assignment (recommended)

1. Replace demo CSVs with real project files:
   - `data/project_equipment_database.csv`
   - `data/external_equipment_list.csv`
2. Add at least 3 new logic rules specific to your engineering standards.
3. Compare red-flag counts before and after improving OCR preprocessing.
4. Build a reviewer checklist for resolving CRITICAL/HIGH flags.

This completes the advanced module on **Spatial Mapping + Data Integration + QA Automation**.

## Section B - Rule-Based Case Library

This section provides a complete **rule-based QA suite** for engineering drawings, organized as Case 1 through Case 14.

How to use this section:
- Run cases in order for a full audit workflow, or run selected cases for targeted reviews.
- Each case produces a structured issue table with severity levels (`CRITICAL`, `HIGH`, `MEDIUM`, `LOW`).
- Each case also saves artifacts (summary, issues, and visualization chart) to support reporting and traceability.

Recommended interpretation workflow:
1. Start from CRITICAL and HIGH findings.
2. Confirm visual/context correctness on the drawing.
3. Decide whether to adjust OCR/detection thresholds, rule parameters, or source data.
4. Re-run affected cases and compare outputs.


In [ ]:
import matplotlib.pyplot as plt

CASE_OUTPUT_DIR = OUTPUT_DIR / "section_b_cases"
CASE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SEVERITY_ORDER = ["CRITICAL", "HIGH", "MEDIUM", "LOW"]


def show_case_results(summary, issues_df, max_rows=20):
    print("Case summary:")
    print(summary)
    if issues_df.empty:
        print("No issues found.")
    else:
        display(issues_df.head(max_rows))


def save_case_outputs(case_id, summary, issues_df):
    case_dir = CASE_OUTPUT_DIR / case_id.lower()
    case_dir.mkdir(parents=True, exist_ok=True)
    with open(case_dir / "summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)
    issues_df.to_csv(case_dir / "issues.csv", index=False, encoding="utf-8")

    sev = issues_df["severity"].value_counts().reindex(SEVERITY_ORDER).fillna(0) if (not issues_df.empty and "severity" in issues_df.columns) else pd.Series([0,0,0,0], index=SEVERITY_ORDER)
    fig = plt.figure(figsize=(7,3.5))
    plt.bar(sev.index, sev.values, color=["#b30000", "#e34a33", "#fdbb84", "#9ecae1"])
    plt.title(case_id + " - Severity Distribution")
    plt.ylabel("Flag count")
    plt.tight_layout()
    fig.savefig(case_dir / "severity_chart.png", dpi=150)
    plt.show()
    plt.close(fig)


def render_case_output_image(case_id, case_name, issues_df, image_rgb, symbols_df=None, associations_df=None, roi=None):
    """Render image-first case output: detections only (no issue-list panel)."""
    canvas = image_rgb.copy()

    # ROI (if case has region constraints)
    if roi is not None:
        x_min, y_min, x_max, y_max = roi
        overlay = canvas.copy()
        cv2.rectangle(overlay, (x_min, y_min), (x_max, y_max), (0, 160, 0), -1)
        canvas = cv2.addWeighted(overlay, 0.12, canvas, 0.88, 0)
        cv2.rectangle(canvas, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)

    # Symbol detections with per-class colors (aligned with Step 4)
    if symbols_df is not None and not symbols_df.empty:
        class_color_local = {}
        for _, s in symbols_df.iterrows():
            cls_id = int(s.get("class_id", -1)) if pd.notna(s.get("class_id", None)) else -1
            cls_name = str(s.get("class_name", "symbol"))

            if cls_id not in class_color_local:
                if "class_color_map" in globals() and cls_id in class_color_map:
                    class_color_local[cls_id] = tuple(int(v) for v in class_color_map[cls_id])
                else:
                    # Deterministic fallback color by class id
                    idx = max(0, cls_id)
                    b = (37 * (idx + 3)) % 256
                    g = (97 * (idx + 5)) % 256
                    r = (157 * (idx + 7)) % 256
                    class_color_local[cls_id] = (int(b), int(g), int(r))

            draw_color = class_color_local[cls_id]
            x1, y1, x2, y2 = int(s["x1"]), int(s["y1"]), int(s["x2"]), int(s["y2"])
            cv2.rectangle(canvas, (x1, y1), (x2, y2), draw_color, 2)
            cv2.putText(canvas, cls_name, (x1, max(15, y1 - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.38, draw_color, 1, cv2.LINE_AA)

    # Text detections + matched links
    if associations_df is not None and not associations_df.empty:
        for _, a in associations_df.iterrows():
            if pd.isna(a.get("text_center_x", None)) or pd.isna(a.get("text_center_y", None)):
                continue
            tx, ty = int(a["text_center_x"]), int(a["text_center_y"])
            txt = str(a.get("text", ""))
            cv2.circle(canvas, (tx, ty), 3, (255, 255, 0), -1)
            cv2.putText(canvas, txt, (tx + 4, max(15, ty - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 0), 1, cv2.LINE_AA)

            if bool(a.get("matched", False)) and pd.notna(a.get("symbol_center_x", None)) and pd.notna(a.get("symbol_center_y", None)):
                sx, sy = int(a["symbol_center_x"]), int(a["symbol_center_y"])
                cv2.line(canvas, (tx, ty), (sx, sy), (0, 255, 255), 1)

    # Minimal header only (no list text block)
    ok = int(len(issues_df) == 0)
    status = "PASS" if ok else "CHECK"
    cv2.putText(canvas, f"{case_id} | {status}", (10, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2, cv2.LINE_AA)
    cv2.putText(canvas, f"Issues: {int(len(issues_df))}", (10, 48), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (230, 230, 230), 1, cv2.LINE_AA)

    plt.figure(figsize=(16, 10))
    plt.imshow(canvas)
    plt.title(f"{case_id} Detection Output")
    plt.axis("off")
    plt.show()

    out_path = CASE_OUTPUT_DIR / case_id.lower() / f"{case_id.lower()}_output.jpg"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(out_path), cv2.cvtColor(canvas, cv2.COLOR_RGB2BGR))
    print("Saved case output image:", out_path)




### Case 1 - Zone-based Symbol + Text Detection

**Goal**: Validate that extracted text-symbol entities stay inside an approved engineering region (ROI).

**What this case checks**:
- Text center must be inside the configured zone boundary.
- Any text outside ROI is flagged as out-of-scope.

**Why this matters**:
- Prevents wrong associations from nearby but unrelated systems.
- Useful when one drawing contains multiple process areas.

**Expected output**:
- `issues_01` with `ZONE_OUT_OF_SCOPE` flags.
- Severity chart and saved case artifacts.


In [ ]:
case_id = "CASE_01_ZONE_BASED"
case_name = "Zone-based symbol + text detection"

# Adjustable detection region (ROI)
x_min = 0
y_min = 0
x_max = int(image_rgb.shape[1] * 0.85)
y_max = int(image_rgb.shape[0] * 0.90)

# Keep only data inside ROI (no outside detection/visualization)
symbols_in_roi = symbols_df[
    (symbols_df["center_x"] >= x_min)
    & (symbols_df["center_x"] <= x_max)
    & (symbols_df["center_y"] >= y_min)
    & (symbols_df["center_y"] <= y_max)
].copy()

associations_in_roi = associations_df[
    (associations_df["text_center_x"] >= x_min)
    & (associations_df["text_center_x"] <= x_max)
    & (associations_df["text_center_y"] >= y_min)
    & (associations_df["text_center_y"] <= y_max)
].copy()

# Case result table only for in-region detections
issues_01 = pd.DataFrame(columns=["severity", "flag_code", "entity_key", "message"])
summary_01 = {
    "case_id": case_id,
    "case_name": case_name,
    "evaluated_rows": int(len(associations_in_roi)),
    "issue_count": 0,
    "roi": {"x_min": x_min, "y_min": y_min, "x_max": x_max, "y_max": y_max},
}

# One visualization output only: ROI + in-region symbols/text/links
case1_viz = image_rgb.copy()
overlay = case1_viz.copy()
cv2.rectangle(overlay, (x_min, y_min), (x_max, y_max), (0, 200, 0), -1)
case1_viz = cv2.addWeighted(overlay, 0.12, case1_viz, 0.88, 0)
cv2.rectangle(case1_viz, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)

# Draw in-region symbol detections only
for _, s in symbols_in_roi.iterrows():
    sx1, sy1, sx2, sy2 = int(s["x1"]), int(s["y1"]), int(s["x2"]), int(s["y2"])
    cls_name = str(s.get("class_name", "symbol"))
    cls_id = int(s.get("class_id", -1)) if pd.notna(s.get("class_id", None)) else -1

    if "class_color_map" in globals() and cls_id in class_color_map:
        box_color = tuple(int(v) for v in class_color_map[cls_id])
    else:
        idx = max(0, cls_id)
        box_color = ((37 * (idx + 3)) % 256, (97 * (idx + 5)) % 256, (157 * (idx + 7)) % 256)

    cv2.rectangle(case1_viz, (sx1, sy1), (sx2, sy2), box_color, 2)
    cv2.putText(case1_viz, cls_name, (sx1, max(15, sy1 - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.38, box_color, 1, cv2.LINE_AA)

# Draw in-region text detections and matched links only
for _, r in associations_in_roi.iterrows():
    tx, ty = int(r["text_center_x"]), int(r["text_center_y"])
    txt = str(r.get("text", ""))

    cv2.circle(case1_viz, (tx, ty), 3, (255, 255, 0), -1)
    cv2.putText(case1_viz, txt, (tx + 4, max(15, ty - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 0), 1, cv2.LINE_AA)

    if bool(r.get("matched", False)) and pd.notna(r.get("symbol_center_x", None)) and pd.notna(r.get("symbol_center_y", None)):
        sx, sy = int(r["symbol_center_x"]), int(r["symbol_center_y"])
        if x_min <= sx <= x_max and y_min <= sy <= y_max:
            cv2.line(case1_viz, (tx, ty), (sx, sy), (0, 255, 255), 1)

plt.figure(figsize=(16, 12))
plt.imshow(case1_viz)
plt.title("Case 1 Output: In-Region Text and Symbol Detections")
plt.axis("off")
plt.show()

case1_image_path = CASE_OUTPUT_DIR / case_id.lower() / "case_01_zone_based_region_output.jpg"
case1_image_path.parent.mkdir(parents=True, exist_ok=True)
cv2.imwrite(str(case1_image_path), cv2.cvtColor(case1_viz, cv2.COLOR_RGB2BGR))
print("Saved Case 1 image:", case1_image_path)



### Case 2 - Valve-only Detection and Association

**Goal**: Focus QA on valve-related tags and symbols only.

**What this case checks**:
- Valve-like text (`V-...` or containing `VALVE`) should map to valve classes.
- Association distance should remain under `MAX_DISTANCE`.

**Why this matters**:
- Reduces review noise for valve discipline checks.
- Catches common mismatch: valve tag linked to non-valve symbol.

**Expected output**:
- `issues_02` with `VALVE_EXPECTED_NON_VALVE_FOUND` and `VALVE_TAG_UNMATCHED`.
- Severity chart and saved case artifacts.


In [ ]:
case_id="CASE_02_VALVE_ONLY"; case_name="Valve-only detection and association"
issues=[]
for _,r in associations_df.iterrows():
    txt=str(r.get("text","")); cls=str(r.get("symbol_class","")); dist=float(r.get("distance_to_symbol",1e9))
    is_valve=("V-" in txt.upper()) or ("VALVE" in txt.upper())
    if is_valve and "VALVE" not in cls.upper():
        issues.append({"severity":"HIGH","flag_code":"VALVE_EXPECTED_NON_VALVE_FOUND","entity_key":txt,"message":"Valve-like tag linked to non-valve class."})
    if is_valve and dist>MAX_DISTANCE:
        issues.append({"severity":"MEDIUM","flag_code":"VALVE_TAG_UNMATCHED","entity_key":txt,"message":"Valve-like tag exceeds distance threshold."})
issues_02=pd.DataFrame(issues); summary_02={"case_id":case_id,"case_name":case_name,"evaluated_rows":int(len(associations_df)),"issue_count":int(len(issues_02))}
show_case_results(summary_02, issues_02); save_case_outputs(case_id, summary_02, issues_02)

render_case_output_image(case_id, case_name, issues_02, image_rgb, symbols_df=valves_df if "valves_df" in globals() else symbols_df, associations_df=associations_df)


### Case 3 - Tag Format and Naming Compliance

**Goal**: Ensure OCR text follows project tag naming conventions.

**What this case checks**:
- Numeric-like text should produce a valid extracted tag.
- Text with numbers but invalid/no tag is flagged.

**Why this matters**:
- Tag format errors break database joins and QA automation.
- Early validation improves downstream reliability.

**Expected output**:
- `issues_03` with `TAG_FORMAT_INVALID`.
- Severity chart and saved case artifacts.


In [ ]:
case_id="CASE_03_TAG_PATTERN"; case_name="Tag format and naming compliance"
issues=[]
for _,r in final_df.iterrows():
    text=str(r.get("text","")); tag=r.get("extracted_tag",None)
    if (tag is None or pd.isna(tag)) and any(ch.isdigit() for ch in text):
        issues.append({"severity":"LOW","flag_code":"TAG_FORMAT_INVALID","entity_key":text,"message":"Numeric text but no valid extracted tag."})
issues_03=pd.DataFrame(issues); summary_03={"case_id":case_id,"case_name":case_name,"evaluated_rows":int(len(final_df)),"issue_count":int(len(issues_03))}
show_case_results(summary_03, issues_03); save_case_outputs(case_id, summary_03, issues_03)

render_case_output_image(case_id, case_name, issues_03, image_rgb, symbols_df=symbols_df, associations_df=associations_df)


### Case 4 - One Tag to One Equipment Uniqueness

**Goal**: Enforce unique mapping between tag and equipment instance.

**What this case checks**:
- A single extracted tag should not map to multiple symbol IDs.
- Duplicate mapping is flagged as high severity.

**Why this matters**:
- Avoids duplicate records and ownership ambiguity.
- Important for equipment registry consistency.

**Expected output**:
- `issues_04` with `DUPLICATE_TAG_MAPPING`.
- Severity chart and saved case artifacts.


In [ ]:
case_id="CASE_04_UNIQUENESS"; case_name="One tag to one equipment uniqueness"
t=final_df[final_df["extracted_tag"].notna()].copy() if "extracted_tag" in final_df.columns else pd.DataFrame()
dup=t.groupby("extracted_tag")["symbol_id"].nunique().reset_index(name="n") if not t.empty else pd.DataFrame(columns=["extracted_tag","n"])
issues_04=pd.DataFrame([{"severity":"HIGH","flag_code":"DUPLICATE_TAG_MAPPING","entity_key":r["extracted_tag"],"message":"Tag mapped to multiple symbols."} for _,r in dup[dup["n"]>1].iterrows()])
summary_04={"case_id":case_id,"case_name":case_name,"evaluated_rows":int(len(final_df)),"issue_count":int(len(issues_04))}
show_case_results(summary_04, issues_04); save_case_outputs(case_id, summary_04, issues_04)

render_case_output_image(case_id, case_name, issues_04, image_rgb, symbols_df=symbols_df, associations_df=associations_df)


### Case 5 - Cross-sheet Line Consistency (Diameter/Spec)

**Goal**: Detect line attribute inconsistency across multiple sheets.

**What this case checks**:
- Same `line_id` across sheets should maintain consistent diameter.

**Why this matters**:
- Cross-sheet inconsistency is a common revision-control problem.
- Helps catch design/document drift early.

**Expected output**:
- `issues_05` with `LINE_DIAMETER_MISMATCH`.
- Severity chart and saved case artifacts.


In [ ]:
case_id="CASE_05_CROSS_SHEET_LINE_CONSISTENCY"; case_name="Cross-sheet line consistency"
issues=[]
if {"line_id","sheet","expected_diameter_mm"}.issubset(set(xref_df.columns)):
    w=xref_df.dropna(subset=["line_id","sheet","expected_diameter_mm"])
    for line_id,g in w.groupby("line_id"):
        if g["sheet"].nunique()>1 and g["expected_diameter_mm"].nunique()>1:
            issues.append({"severity":"HIGH","flag_code":"LINE_DIAMETER_MISMATCH","entity_key":line_id,"message":"Inconsistent diameter across sheets."})
issues_05=pd.DataFrame(issues); summary_05={"case_id":case_id,"case_name":case_name,"evaluated_rows":int(len(xref_df)),"issue_count":int(len(issues_05))}
show_case_results(summary_05, issues_05); save_case_outputs(case_id, summary_05, issues_05)

render_case_output_image(case_id, case_name, issues_05, image_rgb, symbols_df=symbols_df, associations_df=associations_df)


### Case 6 - Service-to-Symbol Compatibility

**Goal**: Validate that detected symbol class is compatible with process service.

**What this case checks**:
- Service labels (e.g., control service) should align with expected symbol families.

**Why this matters**:
- Prevents functionally incorrect symbol assignments.
- Supports process-safety and engineering QA checks.

**Expected output**:
- `issues_06` with `SERVICE_TYPE_INCOMPATIBLE`.
- Severity chart and saved case artifacts.


In [ ]:
case_id="CASE_06_SERVICE_COMPATIBILITY"; case_name="Service-to-symbol compatibility"
issues=[]
if {"service","symbol_class"}.issubset(xref_df.columns):
    for _,r in xref_df.dropna(subset=["service","symbol_class"]).iterrows():
        service=str(r["service"]).lower(); cls=str(r["symbol_class"]).lower()
        if "control" in service and "control" not in cls and "globe" not in cls:
            issues.append({"severity":"HIGH","flag_code":"SERVICE_TYPE_INCOMPATIBLE","entity_key":r.get("extracted_tag",""),"message":"Service may be incompatible with symbol class."})
issues_06=pd.DataFrame(issues); summary_06={"case_id":case_id,"case_name":case_name,"evaluated_rows":int(len(xref_df)),"issue_count":int(len(issues_06))}
show_case_results(summary_06, issues_06); save_case_outputs(case_id, summary_06, issues_06)

render_case_output_image(case_id, case_name, issues_06, image_rgb, symbols_df=symbols_df, associations_df=associations_df)


### Case 7 - Equipment Lifecycle Governance

**Goal**: Prevent invalid references to retired/decommissioned assets.

**What this case checks**:
- If external status is `DECOMMISSIONED`, raise critical flag.

**Why this matters**:
- Decommissioned references are high-risk in operations and maintenance.

**Expected output**:
- `issues_07` with `DECOMMISSIONED_REFERENCE`.
- Severity chart and saved case artifacts.


In [ ]:
case_id="CASE_07_LIFECYCLE"; case_name="Equipment lifecycle governance"
issues_07=pd.DataFrame([{"severity":"CRITICAL","flag_code":"DECOMMISSIONED_REFERENCE","entity_key":r.get("extracted_tag",""),"message":"Drawing references decommissioned equipment."} for _,r in xref_df[xref_df.get("equipment_status",pd.Series([],dtype=str)).astype(str).str.upper()=="DECOMMISSIONED"].iterrows()]) if "equipment_status" in xref_df.columns else pd.DataFrame()
summary_07={"case_id":case_id,"case_name":case_name,"evaluated_rows":int(len(xref_df)),"issue_count":int(len(issues_07))}
show_case_results(summary_07, issues_07); save_case_outputs(case_id, summary_07, issues_07)

render_case_output_image(case_id, case_name, issues_07, image_rgb, symbols_df=symbols_df, associations_df=associations_df)


### Case 8 - Mandatory Annotation Completeness

**Goal**: Ensure required annotation fields are present before approval.

**What this case checks**:
- Missing extracted tag is treated as incomplete annotation.

**Why this matters**:
- Incomplete markup creates manual rework and review delays.

**Expected output**:
- `issues_08` with `MISSING_REQUIRED_TAG`.
- Severity chart and saved case artifacts.


In [ ]:
case_id="CASE_08_MANDATORY_ANNOTATION"; case_name="Mandatory annotation completeness"
issues_08=pd.DataFrame([{"severity":"MEDIUM","flag_code":"MISSING_REQUIRED_TAG","entity_key":r.get("text",""),"message":"Missing extracted tag."} for _,r in final_df.iterrows() if pd.isna(r.get("extracted_tag",np.nan))])
summary_08={"case_id":case_id,"case_name":case_name,"evaluated_rows":int(len(final_df)),"issue_count":int(len(issues_08))}
show_case_results(summary_08, issues_08); save_case_outputs(case_id, summary_08, issues_08)

render_case_output_image(case_id, case_name, issues_08, image_rgb, symbols_df=symbols_df, associations_df=associations_df)


### Case 9 - Distance + Direction Constrained Association

**Goal**: Improve matching quality in dense engineering drawings.

**What this case checks**:
- Match distance should be below threshold.
- Relative text direction should follow drafting expectation.

**Why this matters**:
- Reduces false positives from nearest-neighbor-only matching.

**Expected output**:
- `issues_09` with distance/direction violation flags.
- Severity chart and saved case artifacts.


In [ ]:
case_id="CASE_09_DISTANCE_DIRECTION"; case_name="Distance + direction constrained association"
issues=[]
for _,r in associations_df.iterrows():
    dist=r.get("distance_to_symbol",np.nan)
    if pd.notna(dist) and float(dist)>MAX_DISTANCE:
        issues.append({"severity":"MEDIUM","flag_code":"ASSOCIATION_DISTANCE_EXCEEDED","entity_key":r.get("text",""),"message":"Distance exceeds threshold."})
    tx,sx=r.get("text_center_x",np.nan),r.get("symbol_center_x",np.nan)
    if pd.notna(tx) and pd.notna(sx) and tx<sx:
        issues.append({"severity":"LOW","flag_code":"ASSOCIATION_DIRECTION_VIOLATION","entity_key":r.get("text",""),"message":"Text on unexpected side."})
issues_09=pd.DataFrame(issues); summary_09={"case_id":case_id,"case_name":case_name,"evaluated_rows":int(len(associations_df)),"issue_count":int(len(issues_09))}
show_case_results(summary_09, issues_09); save_case_outputs(case_id, summary_09, issues_09)

render_case_output_image(case_id, case_name, issues_09, image_rgb, symbols_df=symbols_df, associations_df=associations_df)


### Case 10 - Zone Class Whitelist Policy

**Goal**: Enforce allowed symbol classes within a specific scope/zone.

**What this case checks**:
- Detected class not in whitelist is flagged as policy violation.

**Why this matters**:
- Prevents invalid class usage for region-specific standards.

**Expected output**:
- `issues_10` with `FORBIDDEN_CLASS_IN_ZONE`.
- Severity chart and saved case artifacts.


In [ ]:
case_id="CASE_10_ZONE_WHITELIST"; case_name="Zone class whitelist policy"
allowed={c.upper() for c in ["Gate Valve","Ball Valve","Globe Valve","Check Valve","Pressure Gauge"]}
issues_10=pd.DataFrame([{"severity":"HIGH","flag_code":"FORBIDDEN_CLASS_IN_ZONE","entity_key":str(r.get("symbol_class","")),"message":"Class outside zone whitelist."} for _,r in final_df.iterrows() if str(r.get("symbol_class","")) and str(r.get("symbol_class","")).upper() not in allowed])
summary_10={"case_id":case_id,"case_name":case_name,"evaluated_rows":int(len(final_df)),"issue_count":int(len(issues_10))}
show_case_results(summary_10, issues_10); save_case_outputs(case_id, summary_10, issues_10)

render_case_output_image(case_id, case_name, issues_10, image_rgb, symbols_df=symbols_df, associations_df=associations_df)


### Case 11 - Revision Delta Verification (Rev N vs Rev N+1)

**Goal**: Detect changes between drawing revisions.

**What this case checks**:
- New tags in current revision.
- Removed tags compared with baseline revision.

**Why this matters**:
- Supports controlled change tracking and review transparency.

**Expected output**:
- `issues_11` with `UNDECLARED_ADDITION` / `UNDECLARED_REMOVAL`.
- Severity chart and saved case artifacts.


In [ ]:
case_id="CASE_11_REVISION_DELTA"; case_name="Revision delta verification"
baseline_df=final_df.sample(frac=0.75, random_state=42).copy() if len(final_df)>0 else final_df.copy()
cur=set(final_df.get("extracted_tag",pd.Series([],dtype=str)).dropna().astype(str)); base=set(baseline_df.get("extracted_tag",pd.Series([],dtype=str)).dropna().astype(str))
issues=[]
for tag in sorted(cur-base): issues.append({"severity":"MEDIUM","flag_code":"UNDECLARED_ADDITION","entity_key":tag,"message":"Tag appears only in current revision."})
for tag in sorted(base-cur): issues.append({"severity":"MEDIUM","flag_code":"UNDECLARED_REMOVAL","entity_key":tag,"message":"Tag removed from current revision."})
issues_11=pd.DataFrame(issues); summary_11={"case_id":case_id,"case_name":case_name,"evaluated_rows":int(len(final_df)),"issue_count":int(len(issues_11))}
show_case_results(summary_11, issues_11); save_case_outputs(case_id, summary_11, issues_11)

render_case_output_image(case_id, case_name, issues_11, image_rgb, symbols_df=symbols_df, associations_df=associations_df)


### Case 12 - OCR Confidence Governance

**Goal**: Route low-confidence OCR outputs to manual verification.

**What this case checks**:
- OCR confidence below configured threshold is flagged.

**Why this matters**:
- Prevents noisy OCR data from contaminating QA decisions.

**Expected output**:
- `issues_12` with `LOW_OCR_CONFIDENCE_REVIEW_REQUIRED`.
- Severity chart and saved case artifacts.


In [ ]:
case_id="CASE_12_OCR_CONFIDENCE"; case_name="OCR confidence governance"; min_conf=0.45
issues_12=pd.DataFrame([{"severity":"LOW","flag_code":"LOW_OCR_CONFIDENCE_REVIEW_REQUIRED","entity_key":r.get("text",""),"message":"OCR confidence below threshold."} for _,r in final_df[final_df["ocr_confidence"]<min_conf].iterrows()]) if "ocr_confidence" in final_df.columns else pd.DataFrame()
summary_12={"case_id":case_id,"case_name":case_name,"evaluated_rows":int(len(final_df)),"issue_count":int(len(issues_12))}
show_case_results(summary_12, issues_12); save_case_outputs(case_id, summary_12, issues_12)

render_case_output_image(case_id, case_name, issues_12, image_rgb, symbols_df=symbols_df, associations_df=associations_df)


### Case 13 - Critical Equipment Strict Mode

**Goal**: Apply tighter quality controls for high-criticality assets.

**What this case checks**:
- Strict distance threshold for critical assets.
- Type mismatch escalated to critical severity.

**Why this matters**:
- Prioritizes risk-based engineering assurance.

**Expected output**:
- `issues_13` with `CRITICAL_ASSET_RULE_VIOLATION`.
- Severity chart and saved case artifacts.


In [ ]:
case_id="CASE_13_CRITICAL_STRICT"; case_name="Critical equipment strict mode"; strict_distance=120
issues=[]
if {"criticality","distance_to_symbol"}.issubset(xref_df.columns):
    crit=xref_df[xref_df["criticality"].astype(str).str.upper()=="HIGH"]
    for _,r in crit.iterrows():
        if float(r.get("distance_to_symbol",1e9))>strict_distance:
            issues.append({"severity":"CRITICAL","flag_code":"CRITICAL_ASSET_RULE_VIOLATION","entity_key":r.get("extracted_tag",""),"message":"Critical asset exceeds strict distance."})
        if "type_match_ok" in crit.columns and not bool(r.get("type_match_ok",True)):
            issues.append({"severity":"CRITICAL","flag_code":"CRITICAL_ASSET_RULE_VIOLATION","entity_key":r.get("extracted_tag",""),"message":"Critical asset type mismatch."})
issues_13=pd.DataFrame(issues); summary_13={"case_id":case_id,"case_name":case_name,"evaluated_rows":int(len(xref_df)),"issue_count":int(len(issues_13))}
show_case_results(summary_13, issues_13); save_case_outputs(case_id, summary_13, issues_13)

render_case_output_image(case_id, case_name, issues_13, image_rgb, symbols_df=symbols_df, associations_df=associations_df)


### Case 14 - Duplicate Geometry Anomaly Detection

**Goal**: Detect probable duplicate symbol detections.

**What this case checks**:
- Same-class symbols with high IoU and very close centers.

**Why this matters**:
- Prevents double-counting and misleading analytics.

**Expected output**:
- `issues_14` with `POSSIBLE_DUPLICATE_SYMBOL`.
- Severity chart and saved case artifacts.


In [ ]:
case_id="CASE_14_DUPLICATE_GEOMETRY"; case_name="Duplicate geometry anomaly detection"; iou_thr=0.7; center_thr=15

def iou_xyxy(a,b):
    ax1,ay1,ax2,ay2=a; bx1,by1,bx2,by2=b
    ix1,iy1=max(ax1,bx1),max(ay1,by1); ix2,iy2=min(ax2,bx2),min(ay2,by2)
    iw,ih=max(0,ix2-ix1),max(0,iy2-iy1); inter=iw*ih
    area_a=max(0,ax2-ax1)*max(0,ay2-ay1); area_b=max(0,bx2-bx1)*max(0,by2-by1)
    union=area_a+area_b-inter
    return inter/union if union>0 else 0.0

issues=[]; s=symbols_df.reset_index(drop=True)
for i in range(len(s)):
    for j in range(i+1,len(s)):
        if s.loc[i,"class_id"]!=s.loc[j,"class_id"]:
            continue
        a=[s.loc[i,"x1"],s.loc[i,"y1"],s.loc[i,"x2"],s.loc[i,"y2"]]
        b=[s.loc[j,"x1"],s.loc[j,"y1"],s.loc[j,"x2"],s.loc[j,"y2"]]
        iou=iou_xyxy(a,b)
        dc=np.sqrt((s.loc[i,"center_x"]-s.loc[j,"center_x"])**2 + (s.loc[i,"center_y"]-s.loc[j,"center_y"])**2)
        if iou>=iou_thr and dc<=center_thr:
            issues.append({"severity":"MEDIUM","flag_code":"POSSIBLE_DUPLICATE_SYMBOL","entity_key":str(int(s.loc[i,"symbol_id"]))+"-"+str(int(s.loc[j,"symbol_id"])),"message":"High-overlap duplicate candidate."})
issues_14=pd.DataFrame(issues); summary_14={"case_id":case_id,"case_name":case_name,"evaluated_rows":int(len(s)),"issue_count":int(len(issues_14))}
show_case_results(summary_14, issues_14); save_case_outputs(case_id, summary_14, issues_14)

render_case_output_image(case_id, case_name, issues_14, image_rgb, symbols_df=symbols_df, associations_df=associations_df)


### Consolidated Section B Results (All Cases)


In [ ]:
all_summaries=[summary_01,summary_02,summary_03,summary_04,summary_05,summary_06,summary_07,summary_08,summary_09,summary_10,summary_11,summary_12,summary_13,summary_14]
all_issues=[]
for sid, df_ in [("CASE_01_ZONE_BASED",issues_01),("CASE_02_VALVE_ONLY",issues_02),("CASE_03_TAG_PATTERN",issues_03),("CASE_04_UNIQUENESS",issues_04),("CASE_05_CROSS_SHEET_LINE_CONSISTENCY",issues_05),("CASE_06_SERVICE_COMPATIBILITY",issues_06),("CASE_07_LIFECYCLE",issues_07),("CASE_08_MANDATORY_ANNOTATION",issues_08),("CASE_09_DISTANCE_DIRECTION",issues_09),("CASE_10_ZONE_WHITELIST",issues_10),("CASE_11_REVISION_DELTA",issues_11),("CASE_12_OCR_CONFIDENCE",issues_12),("CASE_13_CRITICAL_STRICT",issues_13),("CASE_14_DUPLICATE_GEOMETRY",issues_14)]:
    if not df_.empty:
        t=df_.copy(); t["case_id"]=sid; all_issues.append(t)
master_issues_df=pd.concat(all_issues,ignore_index=True) if all_issues else pd.DataFrame(columns=["case_id","severity","flag_code","entity_key","message"])
summary_df=pd.DataFrame(all_summaries)
display(summary_df); display(master_issues_df.head(50))
summary_path=CASE_OUTPUT_DIR / "section_b_case_summary.csv"; issues_path=CASE_OUTPUT_DIR / "section_b_master_issues.csv"
summary_df.to_csv(summary_path,index=False,encoding="utf-8"); master_issues_df.to_csv(issues_path,index=False,encoding="utf-8")
print("Saved consolidated outputs:"); print("-",summary_path); print("-",issues_path)
